[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/41_segment_ids_from_lengths_solution.ipynb)

# 🟢 Solution: Segment IDs from Lengths

**Primitive: `repeat_interleave`**

**Reduction:** `output[i]` is the sequence index `j` such that `sum(lengths[:j]) <= i < sum(lengths[:j+1])`. In other words, repeat each index `j` exactly `lengths[j]` times.

An equivalent approach using `cumsum`:
1. Build a boundary tensor of zeros with length `total = sum(lengths)`
2. Place `1`s at `cumsum(lengths)[:-1]` (the start of each new segment)
3. `cumsum` the boundary tensor to get the segment IDs

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION — approach 1: repeat_interleave

def segment_ids_from_lengths(lengths: torch.Tensor) -> torch.Tensor:
    # primitive: repeat_interleave — repeats each index j exactly lengths[j] times
    return torch.repeat_interleave(torch.arange(len(lengths), device=lengths.device), lengths)


# ✅ SOLUTION — approach 2: cumsum on boundary markers (equally valid)

def segment_ids_from_lengths_v2(lengths: torch.Tensor) -> torch.Tensor:
    # primitive: cumsum — place 1s at segment boundaries, then cumsum gives the running id
    total = int(lengths.sum().item())
    if total == 0:
        return torch.zeros(0, dtype=torch.long, device=lengths.device)
    boundaries = torch.zeros(total, dtype=torch.long, device=lengths.device)
    starts = lengths.cumsum(0)[:-1]       # start index of each segment (except first)
    boundaries[starts] = 1
    return boundaries.cumsum(0)

In [ ]:
# Verify both approaches
lengths = torch.tensor([3, 1, 4, 2])
print('repeat_interleave:', segment_ids_from_lengths(lengths).tolist())
print('cumsum approach:  ', segment_ids_from_lengths_v2(lengths).tolist())
print('expected:         ', [0, 0, 0, 1, 2, 2, 2, 2, 3, 3])

# Edge cases
print('zeros [0,3,0,1]:  ', segment_ids_from_lengths(torch.tensor([0, 3, 0, 1])).tolist())
print('all-zero lengths: ', segment_ids_from_lengths(torch.tensor([0, 0, 0])).tolist())

In [ ]:
# Run judge
from torch_judge import check
check("segment_ids_from_lengths")